In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

## Read JSON file and convert to RDD

In [5]:
import json

raw_json_rdd = spark.sparkContext.textFile('/content/card_transactions.json')
raw_json_rdd.take(5)

['{"merchant": "M_102", "category": "food", "card_num": "C_108", "user_id": "U_104", "ts": 1579532902, "amount": 243}',
 '{"merchant": "M_103", "category": "cosmetics", "card_num": "C_106", "user_id": "U_103", "ts": 1581759040, "amount": 699}',
 '{"merchant": "M_110", "category": "children", "card_num": "C_104", "user_id": "U_103", "ts": 1584161986, "amount": 228}',
 '{"merchant": "M_110", "category": "groceries", "card_num": "C_107", "user_id": "U_104", "ts": 1584180442, "amount": 795}',
 '{"merchant": "M_106", "category": "food", "card_num": "C_106", "user_id": "U_103", "ts": 1579077866, "amount": 722}']

In [6]:
# json.loads converts json string to dict
# 1580515200 <= ts < 1583020800 means 1st Feb to 28th Feb 2020
dict_rdd = raw_json_rdd.map(lambda x: json.loads(x)).filter(lambda x: x if (x['ts'] >= 1580515200 and x['ts'] < 1583020800) else None).cache()
dict_rdd.take(2)

[{'merchant': 'M_103',
  'category': 'cosmetics',
  'card_num': 'C_106',
  'user_id': 'U_103',
  'ts': 1581759040,
  'amount': 699},
 {'merchant': 'M_102',
  'category': 'food',
  'card_num': 'C_101',
  'user_id': 'U_101',
  'ts': 1581758143,
  'amount': 855}]

## Get distinct list of categories in which user has made expenditure

### Without using combineByKey()

In [7]:
def GetCategoryPerUser(tran_iter):
  for transaction in tran_iter:
    user_id = transaction.get('user_id')
    category = transaction.get('category')
    yield (user_id, category)


def getDistinctCategory(val1, val2):
  if isinstance(val1, list):
    if isinstance(val2, list):
      for val in val2:
        if val not in val1:
          val1.append(val)
    else:
      if val2 not in val1:
        val1.append(val2)
  else:
    if val1 != val2:
      val1 = [val1, val2]
  return val1

In [8]:
dict_rdd.mapPartitions(GetCategoryPerUser).reduceByKey(getDistinctCategory).mapValues(lambda val: ", ".join(val)).collect()

[('U_103', 'cosmetics, groceries, entertainment, food, children'),
 ('U_104', 'groceries, entertainment, food, children, cosmetics'),
 ('U_101', 'food, cosmetics, groceries, children, entertainment'),
 ('U_102', 'children, groceries, entertainment, food, cosmetics')]

### Using combineByKey()

In [ ]:
def createCombiner(v):
  return set([v])

def mergeValue(c, v):
  c.add(v)
  return c

def mergeCombiners(c1, c2):
  c1.update(c2)
  return ", ".join(c1)

In [10]:
dict_rdd.mapPartitions(GetCategoryPerUser).combineByKey(createCombiner, mergeValue, mergeCombiners).take(5)

[('U_103', 'groceries, food, entertainment, cosmetics, children'),
 ('U_104', 'groceries, food, entertainment, cosmetics, children'),
 ('U_101', 'groceries, food, entertainment, cosmetics, children'),
 ('U_102', 'groceries, food, entertainment, cosmetics, children')]